# Final Assignment: training

This notebook contains the source code for training the weights of the classification model.

## Download dataset

The following cell needs to be ran only once. It downloads and unzips the data. The unzip CLI tool might not be installed by default on Windows, so if you are using windows, please unzip manually into a folder called "data", or see the README.md.

In [2]:
# Downloading the dataset (unzip might not be installed on Windows)
!curl -L -o ./data.zip https://www.kaggle.com/api/v1/datasets/download/navoneel/brain-mri-images-for-brain-tumor-detection
!unzip -d ./data ./data.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 15.0M  100 15.0M    0     0  8978k      0  0:00:01  0:00:01 --:--:-- 17.8M
Archive:  ./data.zip
replace ./data/brain_tumor_dataset/no/1 no.jpeg? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C


## Importing libraries

See README.md for installing dependencies :).

In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
from torchvision import models, datasets, transforms
from tqdm import tqdm

## Setting the seed

Setting the seed for good reproducability :D

In [2]:
torch.manual_seed(21)  # Using my lucky number :D
np.random.seed(21)

## Setting the device used for training

In [33]:
device = "cpu"
if torch.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
print(f"Using: {device}")

Using: mps


## Preparing the dataset

In [34]:
transform = transforms.Compose(
    [
        transforms.Resize((299, 299)),  # 299 is the minimal image size of InceptionV3
        transforms.ToTensor()
    ]
)

In [35]:
path = os.path.join("data", "brain_tumor_dataset")

In [36]:
dataset = datasets.ImageFolder(
    root=path,
    transform=transform
)
train_dataset, test_dataset = random_split(
    dataset=dataset,
    lengths=[0.8, 0.2]
)

In [37]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

## Creating the model

### Loading pre-trained model

In [39]:
model = models.inception_v3(weights=models.Inception_V3_Weights).to(device)

### Adding custom classification head

In [40]:
model.fc = nn.Linear(in_features=2048, out_features=2)  # Got the input features from the sourcecode hihi

### Freezing layers

## Training the model

### Setting hyper parameters

In [41]:
EPOCHS = 100
LR = 1e-3

### Initializing optimizer and loss function

In [42]:
optimizer = Adam(params=model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss().to(device)

### Setting model to train mode and moving to device

In [44]:
model = model
model.train()

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

In [46]:
for epoch in range(EPOCHS):
    running_loss = 0

    for X, y in tqdm(train_loader, f"Epoch {epoch +1}"):
        X = X.to(device)
        y = y.to(device)

        y_pred = model(X)[0]
        loss = criterion(y_pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch complete: {epoch + 1}/{EPOCHS} (loss: {running_loss})")

Epoch 1:  23%|██▎       | 3/13 [00:54<03:02, 18.27s/it]


KeyboardInterrupt: 